In [1]:
import pandas as pd
import numpy as np
import os
from scipy.sparse import csr_matrix
from implicit import als

BASE_DIR = os.path.dirname(os.path.dirname(os.path.abspath("02_collaborative_filtering.ipynb")))
ruta = os.path.join(BASE_DIR, "data", "processed", "ratings_filtrado.csv")

df = pd.read_csv(ruta)
print(f"Dataset cargado: {df.shape}")
print(df.head())

c:\Users\diego\OneDrive\Desktop\motor recomendaciones personalizado\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Dataset cargado: (394908, 4)
           userId   productId  rating   timestamp
0  A274NIJWOQWE30  1304351475     5.0  1385251200
1   AN33X95J5460X  1304351475     4.0  1395187200
2  A169NC0ZW6XKRD  1304482685     3.0  1391558400
3  A22ZFXQE8AWPEP  1304482685     1.0  1383177600
4  A22VW0P4VZHDE3  1304482685     5.0  1384128000


In [2]:
# Convertir usuarios y productos a índices numéricos
df["user_idx"] = pd.Categorical(df["userId"]).codes
df["product_idx"] = pd.Categorical(df["productId"]).codes

print(f"Usuarios únicos: {df['user_idx'].nunique():,}")
print(f"Productos únicos: {df['product_idx'].nunique():,}")

# Crear matriz sparse usuario-producto
# Las filas son productos, las columnas son usuarios (así lo requiere implicit)
matriz = csr_matrix(
    (df["rating"].values, (df["product_idx"].values, df["user_idx"].values))
)

print(f"\nMatriz sparse: {matriz.shape}")
print(f"Valores no cero: {matriz.nnz:,}")
print(f"Densidad: {matriz.nnz / (matriz.shape[0] * matriz.shape[1]) * 100:.3f}%")

Usuarios únicos: 52,204
Productos únicos: 57,289

Matriz sparse: (57289, 52204)
Valores no cero: 394,908
Densidad: 0.013%


In [3]:
# Entrenar modelo ALS (Alternating Least Squares)
modelo = als.AlternatingLeastSquares(
    factors=50,        # dimensiones del embedding
    iterations=20,     # épocas de entrenamiento
    regularization=0.1,
    alpha=40,
    use_gpu=False
)

print("Entrenando modelo ALS...")
modelo.fit(matriz)
print("✓ Modelo entrenado")

c:\Users\diego\OneDrive\Desktop\motor recomendaciones personalizado\.venv\Lib\site-packages\implicit\cpu\als.py:95: RuntimeWarning: OpenBLAS is configured to use 8 threads. It is highly recommended to disable its internal threadpool by setting the environment variable 'OPENBLAS_NUM_THREADS=1' or by calling 'threadpoolctl.threadpool_limits(1, "blas")'. Having OpenBLAS use a threadpool can lead to severe performance issues here.
  check_blas_config()


Entrenando modelo ALS...


100%|██████████| 20/20 [00:03<00:00,  5.06it/s]

✓ Modelo entrenado


In [11]:
from scipy.sparse import csr_matrix
from implicit import als

# Mapeos fijos
user_cat = pd.Categorical(df["userId"])
product_cat = pd.Categorical(df["productId"])

user_to_idx = dict(enumerate(user_cat.categories))
product_to_idx = dict(enumerate(product_cat.categories))

user_idx_map = {v: k for k, v in user_to_idx.items()}
product_idx_map = {v: k for k, v in product_to_idx.items()}

df["u_idx"] = df["userId"].map(user_idx_map)
df["p_idx"] = df["productId"].map(product_idx_map)

# Matriz: filas=productos, columnas=usuarios
matriz2 = csr_matrix(
    (df["rating"].values, (df["p_idx"].values, df["u_idx"].values)),
    shape=(len(product_to_idx), len(user_to_idx))
)

print(f"Matriz: {matriz2.shape}")

# Reentrenar con esta matriz
modelo2 = als.AlternatingLeastSquares(
    factors=50, iterations=20, regularization=0.1, alpha=40, use_gpu=False
)
modelo2.fit(matriz2)
print("✓ Modelo entrenado")

# Función de recomendación
def recomendar(user_id, n=10):
    if user_id not in user_idx_map:
        print("Usuario no encontrado")
        return
    
    u_idx = user_idx_map[user_id]
    user_items = csr_matrix(matriz2.T)[u_idx]
    
    ids, scores = modelo2.recommend(u_idx, user_items, N=n, filter_already_liked_items=True)
    
    print(f"Recomendaciones para {user_id}:\n")
    for i, (idx, score) in enumerate(zip(ids, scores), 1):
        print(f"{i}. {product_to_idx[idx]} | Score: {score:.4f}")

# Probar
recomendar(df["userId"].iloc[0])

Matriz: (57289, 52204)


100%|██████████| 20/20 [00:04<00:00,  4.16it/s]

✓ Modelo entrenado


IndexError: index 52221 is out of bounds for axis 1 with size 52204

In [12]:
# Diagnóstico
print(f"Total usuarios en mapeo: {len(user_idx_map)}")
print(f"Total productos en mapeo: {len(product_idx_map)}")
print(f"Max u_idx en df: {df['u_idx'].max()}")
print(f"Max p_idx en df: {df['p_idx'].max()}")
print(f"Forma matriz2: {matriz2.shape}")
print(f"Factors del modelo: {modelo2.user_factors.shape}")
print(f"Item factors del modelo: {modelo2.item_factors.shape}")

usuario_prueba = df["userId"].iloc[0]
print(f"\nUsuario prueba: {usuario_prueba}")
print(f"Índice del usuario: {user_idx_map[usuario_prueba]}")

Total usuarios en mapeo: 52204
Total productos en mapeo: 57289
Max u_idx en df: 52203
Max p_idx en df: 57288
Forma matriz2: (57289, 52204)
Factors del modelo: (57289, 50)
Item factors del modelo: (52204, 50)

Usuario prueba: A274NIJWOQWE30
Índice del usuario: 16503


In [13]:
# Matriz correcta: filas=usuarios, columnas=productos
matriz3 = csr_matrix(
    (df["rating"].values, (df["u_idx"].values, df["p_idx"].values)),
    shape=(len(user_idx_map), len(product_idx_map))
)

print(f"Matriz corregida: {matriz3.shape}  ← debe ser (usuarios, productos)")

modelo3 = als.AlternatingLeastSquares(
    factors=50, iterations=20, regularization=0.1, alpha=40, use_gpu=False
)
modelo3.fit(matriz3)
print(f"User factors: {modelo3.user_factors.shape}")
print(f"Item factors: {modelo3.item_factors.shape}")
print("✓ Modelo entrenado correctamente")

def recomendar(user_id, n=10):
    if user_id not in user_idx_map:
        print("Usuario no encontrado")
        return
    
    u_idx = user_idx_map[user_id]
    user_items = matriz3[u_idx]

    ids, scores = modelo3.recommend(u_idx, user_items, N=n, filter_already_liked_items=True)
    
    print(f"\nRecomendaciones para {user_id}:\n")
    for i, (idx, score) in enumerate(zip(ids, scores), 1):
        print(f"{i}. {product_to_idx[idx]} | Score: {score:.4f}")

recomendar(df["userId"].iloc[0])

Matriz corregida: (52204, 57289)  ← debe ser (usuarios, productos)


100%|██████████| 20/20 [00:04<00:00,  4.99it/s]

User factors: (52204, 50)
Item factors: (57289, 50)
✓ Modelo entrenado correctamente

Recomendaciones para A274NIJWOQWE30:

1. B001MA0QY2 | Score: 1.4002
2. B00538TSMU | Score: 1.3165
3. B006L1DNWY | Score: 1.2730
4. B007BLN17K | Score: 1.2589
5. B00132ZG3U | Score: 1.2573
6. B004OHQR1Q | Score: 1.2529
7. B0069SC0OQ | Score: 1.2342
8. B0068Y6CA4 | Score: 1.2297
9. B00A51LI1O | Score: 1.2287
10. B006SVCY6I | Score: 1.2254


In [14]:
import pickle
import os

modelos_dir = os.path.join(BASE_DIR, "data", "processed")

# Guardar modelo
with open(os.path.join(modelos_dir, "modelo_als.pkl"), "wb") as f:
    pickle.dump(modelo3, f)

# Guardar mapeos
with open(os.path.join(modelos_dir, "mapeos.pkl"), "wb") as f:
    pickle.dump({
        "user_to_idx": user_idx_map,
        "product_to_idx": product_to_idx,
        "idx_to_product": product_to_idx
    }, f)

print("✓ Modelo guardado en data/processed/modelo_als.pkl")
print("✓ Mapeos guardados en data/processed/mapeos.pkl")

✓ Modelo guardado en data/processed/modelo_als.pkl
✓ Mapeos guardados en data/processed/mapeos.pkl


In [15]:
# Ver los productos más populares que necesitan descripción
productos_populares = df.groupby("productId").size().sort_values(ascending=False).head(20)
print("Top 20 productos más reseñados:")
print(productos_populares)
print(f"\nTotal productos únicos: {df['productId'].nunique():,}")

Top 20 productos más reseñados:
productId
B0043OYFKU    539
B000ZMBSPE    539
B004OHQR1Q    518
B000142FVW    458
B0069FDR96    453
B00150LT40    443
B001MA0QY2    434
B003V265QW    416
B006L1DNWY    379
B008U1Q4DI    363
B007BLN17K    360
B001JKTTVQ    339
B005BF1M10    327
B000UVZU1S    317
B0030O3VRW    310
B004D24818    307
B002MZ8BK2    298
B00016XJ4M    294
B005C1C02S    290
B00538TSMU    283
dtype: int64

Total productos únicos: 57,289


In [17]:
import random
import pandas as pd

random.seed(42)

categorias = [
    "moisturizing face cream with SPF 30 for daily use",
    "anti-aging serum with retinol and vitamin C",
    "volumizing shampoo for thin and fine hair",
    "long-lasting matte lipstick with hydrating formula",
    "natural mineral foundation with buildable coverage",
    "gentle exfoliating face scrub with micro beads",
    "nourishing hair conditioner for dry and damaged hair",
    "waterproof mascara for lengthening and volumizing",
    "brightening eye cream for dark circles and puffiness",
    "refreshing toner with hyaluronic acid and niacinamide",
    "hydrating lip balm with SPF 15 and vitamin E",
    "clarifying face wash for oily and acne-prone skin",
    "intensive hair mask with keratin and argan oil",
    "setting powder for a matte and long-lasting finish",
    "rose hip oil for skin regeneration and hydration"
]

# Generar descripción para cada producto único
productos_unicos = df["productId"].unique()
descripciones = {
    producto: random.choice(categorias) 
    for producto in productos_unicos
}

df_productos = pd.DataFrame([
    {"productId": pid, "descripcion": desc} 
    for pid, desc in descripciones.items()
])

print(f"Productos con descripción: {len(df_productos):,}")
print(df_productos.head(10))

# Guardar
df_productos.to_csv(
    os.path.join(BASE_DIR, "data", "processed", "productos_descripciones.csv"), 
    index=False
)
print("\n✓ Descripciones guardadas")

Productos con descripción: 57,289
    productId                                        descripcion
0  1304351475       hydrating lip balm with SPF 15 and vitamin E
1  1304482685        anti-aging serum with retinol and vitamin C
2  1403790965  moisturizing face cream with SPF 30 for daily use
3  1412759676  clarifying face wash for oily and acne-prone skin
4  3227001381  natural mineral foundation with buildable cove...
5  535795531X  long-lasting matte lipstick with hydrating for...
6  535795545X  long-lasting matte lipstick with hydrating for...
7  5357955867          volumizing shampoo for thin and fine hair
8  5357955948  clarifying face wash for oily and acne-prone skin
9  5357956111        anti-aging serum with retinol and vitamin C

✓ Descripciones guardadas
